In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/nextgen_nlp_final"
ARTIFACTS_DIR = f"{PROJECT_ROOT}/artifacts"

CLEAN_V2_PATH = f"{ARTIFACTS_DIR}/clean_news_v2_light_regex.parquet"
print("Exists:", os.path.exists(CLEAN_V2_PATH), CLEAN_V2_PATH)

df = pd.read_parquet(CLEAN_V2_PATH)
print("Shape:", df.shape)
print(df.columns.tolist())
df.head(2)

Mounted at /content/drive
Exists: True /content/drive/MyDrive/nextgen_nlp_final/artifacts/clean_news_v2_light_regex.parquet
Shape: (197985, 3)
['date', 'title_clean', 'text_clean']


,date,title_clean,text_clean
0,2025-06-23 00:00:00+00:00,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,2024-07-01 00:00:00+00:00,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...


In [2]:
DATE_COL = "date"
TITLE_COL = "title_clean"
TEXT_COL = "text_clean"

安装 + 加载 sentence-transformers

In [3]:
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2) Query embedding（相关性定义）

In [4]:
query = """
Articles discussing how artificial intelligence impacts industries, companies, workforce,
productivity, adoption, automation, regulation, business transformation, and enterprise deployment.
"""
query_emb = model.encode([query], normalize_embeddings=True)  # shape (1, dim)

3) 分块计算 semantic score（关键：不构造全量 combo_text，不用 pandas .str）

In [5]:
import math

def iter_text_batches(df, title_col, text_col, batch_size=256, max_chars=1000):
    n = len(df)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        titles = df[title_col].iloc[start:end].astype(str).tolist()
        texts  = df[text_col].iloc[start:end].astype(str).tolist()
        batch = [(t + " " + x[:max_chars]).strip() for t, x in zip(titles, texts)]
        yield start, end, batch

batch_size = 256
scores = np.empty(len(df), dtype=np.float32)

for start, end, batch in iter_text_batches(df, TITLE_COL, TEXT_COL, batch_size=batch_size, max_chars=1000):
    emb = model.encode(batch, normalize_embeddings=True)  # (bs, dim)
    # cosine sim = dot because normalized
    s = emb @ query_emb.T                                  # (bs, 1)
    scores[start:end] = s.squeeze(1).astype(np.float32)
    if (start // batch_size) % 200 == 0:
        gc.collect()

df["semantic_score"] = scores
print("Done. Score stats:", float(df["semantic_score"].min()), float(df["semantic_score"].mean()), float(df["semantic_score"].max()))

Done. Score stats: -0.13739310204982758 0.26581013202667236 0.7510455846786499


4) 过滤：先用 percentile（简单、可解释）- 换成前70%

In [6]:
keep_top = 0.70
threshold = np.quantile(df["semantic_score"], 1 - keep_top)
df_rel = df[df["semantic_score"] >= threshold].copy()

print("Before:", df.shape)
print("After semantic filtering:", df_rel.shape)
print("Keep rate:", len(df_rel)/len(df))
print("Threshold:", float(threshold))

Before: (197985, 4)
After semantic filtering: (138589, 4)
Keep rate: 0.6999974745561532
Threshold: 0.19626334309577942


5) 保存 relevant 数据集（artifact）

In [7]:
RELEVANT_PATH = f"{ARTIFACTS_DIR}/news_relevant_v1.parquet"
df_rel[[DATE_COL, TITLE_COL, TEXT_COL, "semantic_score"]].to_parquet(RELEVANT_PATH, index=False)

print("Saved:", RELEVANT_PATH)

Saved: /content/drive/MyDrive/nextgen_nlp_final/artifacts/news_relevant_v1.parquet


顺手写一个 report（PPT / write-up 直接用）：

In [8]:
import json

report = {
    "input": os.path.basename(CLEAN_V2_PATH),
    "output": os.path.basename(RELEVANT_PATH),
    "rows_before": int(len(df)),
    "rows_after": int(len(df_rel)),
    "keep_rate": float(len(df_rel)/len(df)),
    "threshold_quantile": float(1-keep_top),
    "threshold_value": float(threshold),
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "max_chars_used": 1000,
    "batch_size": batch_size
}

REPORT_PATH = f"{ARTIFACTS_DIR}/semantic_filter_report_v1.json"
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("Saved report:", REPORT_PATH)

Saved report: /content/drive/MyDrive/nextgen_nlp_final/artifacts/semantic_filter_report_v1.json


6) 快速 sanity check：看 3 条结果像不像“行业影响”文章

In [9]:
pd.set_option("display.max_colwidth", 500)
df_rel.sample(3, random_state=0)[[DATE_COL, TITLE_COL, "semantic_score"]]

,date,title_clean,semantic_score
120839,2025-04-22 00:00:00+00:00,"Questflow Labs Selected For ‘Google For Startups Accelerator: AI First’ Program, Gaining Access To Support And Expert Mentorship | Metaverse Post",0.296593
109834,2025-08-02 00:00:00+00:00,"Meta Platforms Sells $2 Billion in Data Center Assets to Fund AI Infrastructure, ETCIO",0.450540
183632,2023-09-12 00:00:00+00:00,"Aspirion Acquires Infinia ML, an established leader in AI and Machine Learning",0.292604
